# Exploratory Notebook : Chunking and Embedding
## Introduction 

After cleaning the first dataset in the first exploratory notebook, we can move on to chunking and embedding this data. We will take the collected text, break it down into chunks, and then convert it into vectors. These vectors will allow our RAG system to quickly compare the questions it is asked with these text chunks to find an answer. 

In [18]:
import json 
from pathlib import Path 
import pandas as pd 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import torch

## 01. Chunking

Chunking involves breaking our text down into smaller pieces.

Let's start by loading our text

In [2]:
with open(Path("data/processed") / "documents_01.json", encoding="utf-8") as f:
    documents = json.load(f)

for name, (text, starts) in documents.items():
    print(f"{name} : {len(text)} character and {len(starts)} pages")

avis_ASNR_2026.pdf : 10181 character and 4 pages
avis_expertise_ASNR_2026.pdf : 20448 character and 6 pages
dossier_concertation_calogena_2026.pdf : 156449 character and 39 pages


We're going to use LangChain to perform chunking

[LangChain RecursiveCharacterTextSplitter documentation](https://lagnchain.readthedocs.io/en/stable/modules/indexes/text_splitters/examples/recursive_text_splitter.html)

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap  = 150,
    add_start_index = True, # We note witch character starting the chunk
)

We're now going to create a function to track the page number associated with our text segments. 
The idea is: 
- We start with the first character, which is associated with the first page. Tthis serves as our reference point. 
- We iterate over each character at the beginning of each chunk, asking each time “Has the page changed?”
- If so, we associate the new characters with the new page.

In [8]:
def find_page(character_number : int, starting_page : list) -> int :
    """
    Function that returns the page number associated with the chunk.
    character_number : int. character position
    starting_page : list. list with the character and page number.
    """
    page = starting_page[0][1] # 0 the character and 1 the associated page

    for start_character, number_page in starting_page:
        if start_character <= character_number: # We want the page that corresponds to the current character. So, among the pages that have been opened, we want the most recently opened one.
            page = number_page
    return page 

We're now going to build our dataframe of chunks

In [10]:
rows = []

for origin, (text, page_starts) in documents.items():
    chunks = splitter.create_documents([text])

    for chunk in chunks:
        start = chunk.metadata["start_index"]
        end = start + len(chunk.page_content) -1 

        rows.append({
            "origin" : origin,
            "start" : start,
            "end" : end,
            "page_start" : find_page(start, page_starts),
            "page_end" : find_page(end, page_starts),
            "text" : chunk.page_content,
        })

df_chunks = pd.DataFrame(rows)

We will now use a model that will: 
- tokenize 
- feed the text into its pre-trained neural network to determine the meaning of the sentences 
- create a list of vectors that will allow us to compare the meaning of the questions with that of the text segments. 

We're going to choose this one [multilingual-e5-small](https://huggingface.co/intfloat/multilingual-e5-small) because: 
- It's small, free, and local. It will allow for a quick search, with no data leaving the user's computer. 
- It's well known for route searches. 
- It's multilingual.

So we'll call it and build the dictionary it requests

In [12]:
model = SentenceTransformer("intfloat/multilingual-e5-small")

texts = []
for text in df_chunks["text"]:
    texts.append("passage: " + text)

embeddings = model.encode(texts, normalize_embeddings=True)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5378.01it/s]


Let's take a look at a chunk

In [13]:
print(df_chunks.loc[10, "text"])

L’ASNR considère que c es intentions présentées au stade des options de sûreté vont dans le sens d’une réduction du risque de blocage par mode commun des deux systèmes d’arrêt d’urgence.
L’absence de diversification fonctionnelle des parties mobiles sollicitées à l’étape 3 est acceptable dans son principe. Néanmoins, cette option de sûreté nécessitera d’être consolidée par des éléments détaillés de conception, de qualification et d’essais.
L’ASNR considère que, dans le cadre d’une éventuelle demande d’autorisation de création, CALOGENA devra pr ésenter des éléments détaillés concernant leur dimensionnement effectif ainsi que des éléments justificatifs de la représentativité du programme de qualification prévu, en particulier des essais, pour assurer leur opérabilité en situations incidentelles et accidentelles, y compris en cas de séisme extrême ou de chute d’aéronef.


The chunk is good, the content is coherent and revolves around the same topic 

Now let's see if it correctly associates the pages with the chunk

In [17]:
chunk = df_chunks.sample(1).iloc[0]
print(chunk["origin"], "pages", chunk["page_start"], "to", chunk["page_end"])

dossier_concertation_calogena_2026.pdf pages 33 to 33


This track is definitely on the right page. So it worked.

For that, we'll use this documentation: [semantic search documentation](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html)

In [23]:
queries = [
    "Le réacteur CAL-30 a quelle puissance ?", # Deliberately simple
    "Les deux systèmes d'arrêt d'urgence du réacteur CAL-30 sont jugés comment par l'ASNR ? ", # more complex
    "Quels risques de défaillance sont enoncés par l'ASNR et que devra donc montrer Calogena pour une autorisation de création ?" # more complex
]

top_k = 3
for query in queries:
    query_embedding = model.encode("query: " + query, normalize_embeddings=True)

    similarity_scores = model.similarity(query_embedding, embeddings)[0]
    scores, indices = torch.topk(similarity_scores, k=top_k)
    corpus = df_chunks["text"].tolist()

    print("\nQuery:", query)
    print(f"Top {top_k} most similar chunks:")

    for score, idx in zip(scores, indices):
        print(f"(Score: {score:.4f})", corpus[idx])



Query: Le réacteur CAL-30 a quelle puissance ?
Top 3 most similar chunks:
(Score: 0.8787) 1. DESCRIPTION DU PROJET DE RÉACTEUR 1.1. PRINCIPES GÉNÉRAUX DE CONCEPTION DU RÉACTEUR CAL-30 D’une puissance nominale de 30 MWth, le réacteur CAL-30 est un petit réacteur modulaire à eau légère fonctionnant à basse pression et basse température, conçu pour fournir de la chaleur à des réseaux de chaleur urbains (RCU).
Installée dans une piscine d’eau légère, la chaudière nucléaire est notamment composée de la cuve du réacteur, de son couvercle et de l’échangeur de chaleur primaire/secondaire. Le cœur du réacteur, d’un volume de l’ordre du mètre cube, est situé en partie basse de la cuve. Les assemblages combustibles (AC) le constituant sont d’une technologie similaire à celle utilisée dans les réacteurs à eau sous pression (REP), mais de hauteur plus petite (de l’ordre du mètre).

1 L’article 3.1 de l’arrêté [2] mentionne à son alinéa II que : « La mise en œuvre du principe de défense en profonde